# HR Employee Attrition 데이터 탐색

목표: 컬럼 구조, 결측치, 타겟(`Attrition`) 분포를 확인하고 SHAP 분석에 쓸 주요 특성을 추린다.

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/HR-Employee-Attrition.csv")
df.shape

(1470, 35)

In [2]:
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [3]:
df.dtypes

Age                         int64
Attrition                     str
BusinessTravel                str
DailyRate                   int64
Department                    str
DistanceFromHome            int64
Education                   int64
EducationField                str
EmployeeCount               int64
EmployeeNumber              int64
EnvironmentSatisfaction     int64
Gender                        str
HourlyRate                  int64
JobInvolvement              int64
JobLevel                    int64
JobRole                       str
JobSatisfaction             int64
MaritalStatus                 str
MonthlyIncome               int64
MonthlyRate                 int64
NumCompaniesWorked          int64
Over18                        str
OverTime                      str
PercentSalaryHike           int64
PerformanceRating           int64
RelationshipSatisfaction    int64
StandardHours               int64
StockOptionLevel            int64
TotalWorkingYears           int64
TrainingTimesL

In [4]:
df.isna().sum().sort_values(ascending=False)

Age                         0
Attrition                   0
BusinessTravel              0
DailyRate                   0
Department                  0
DistanceFromHome            0
Education                   0
EducationField              0
EmployeeCount               0
EmployeeNumber              0
EnvironmentSatisfaction     0
Gender                      0
HourlyRate                  0
JobInvolvement              0
JobLevel                    0
JobRole                     0
JobSatisfaction             0
MaritalStatus               0
MonthlyIncome               0
MonthlyRate                 0
NumCompaniesWorked          0
Over18                      0
OverTime                    0
PercentSalaryHike           0
PerformanceRating           0
RelationshipSatisfaction    0
StandardHours               0
StockOptionLevel            0
TotalWorkingYears           0
TrainingTimesLastYear       0
WorkLifeBalance             0
YearsAtCompany              0
YearsInCurrentRole          0
YearsSince

In [5]:
df["Attrition"].value_counts(normalize=True)

Attrition
No     0.838776
Yes    0.161224
Name: proportion, dtype: float64

## 클래스 불균형

`Attrition`(퇴사) = Yes 비율이 약 16%로 뚜렷한 클래스 불균형이 있음. baseline 정확도가 높게 나와도 다수 클래스(No)만 맞춰서 그런 것일 수 있으니, 다음 세션에서 accuracy 외에 recall/precision도 같이 봐야 함.

In [6]:
# 결측치는 없지만, 상수 컬럼(정보 없음)과 ID성 컬럼이 섞여 있는지 확인
n = len(df)
const_cols = [c for c in df.columns if df[c].nunique() <= 1]
id_like_cols = [c for c in df.columns if df[c].nunique() == n]
print("상수 컬럼:", const_cols)
print("ID성 컬럼:", id_like_cols)

상수 컬럼: ['EmployeeCount', 'Over18', 'StandardHours']
ID성 컬럼: ['EmployeeNumber']


→ `EmployeeCount`(전부 1), `Over18`(전부 Y), `StandardHours`(전부 80)는 상수라 정보가 없고, `EmployeeNumber`는 행마다 고유한 ID. 셋 다 `common.py`의 자동 상수/ID 제거 로직에서 걸러진다 (컬럼명을 직접 나열하지 않아도 됨).

In [7]:
useful_cols = [c for c in df.columns if c not in ["Attrition"] + const_cols + id_like_cols]
categorical_like = [c for c in useful_cols if df[c].dtype == "object"]
numeric_like = [c for c in useful_cols if c not in categorical_like]
print("범주형:", categorical_like)
print("수치형:", numeric_like)

범주형: []
수치형: ['Age', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']


## SHAP 분석에 쓸 주요 특성 5~8개

1. **OverTime** — 야근 여부, Attrition 데이터셋에서 가장 강한 신호로 잘 알려짐
2. **MonthlyIncome** — 급여 수준
3. **Age** — 연령
4. **JobSatisfaction** — 직무 만족도
5. **YearsAtCompany** — 근속 연수
6. **TotalWorkingYears** — 총 경력
7. **JobRole** — 직무
8. **WorkLifeBalance** — 워라밸 만족도

`EmployeeCount`, `Over18`, `StandardHours`, `EmployeeNumber`는 상수/ID성이라 제외.